# TEP fault detection: Colab runner

Runs `tep_pipeline.py` from this repo on the dataset stored in your Google Drive.

1. **Setup** mounts Drive and fetches the latest code from GitHub (re-run it to pick up changes).
2. **Settings**: point `DATA_DIR` at the Drive folder holding the four TEP CSVs.
3. **Train** runs every model family listed in `FAMILIES`. Finished models are saved to Drive, so if Colab disconnects, just run it again.
4. **Results** shows the validation tables. **Final test** runs once, at the very end.
5. **Download** zips the website files.

In [ ]:
# Setup: mount Drive and get the latest code
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!git clone -q https://github.com/Fang099/tep-fault-detection.git 2>/dev/null || git -C /content/tep-fault-detection pull -q
%cd /content/tep-fault-detection
!git log -1 --format="Code version: %h (%cr) %s"

In [ ]:
# Settings
DATA_DIR = "/content/drive/MyDrive/TEP"     # folder with the 4 CSVs (results are saved here too)
RUNS_PER_FAULT = 100                         # 0 = all 500 runs per fault
FAMILIES = "logreg,pcalr,pcamon,tree,rf,gb,ada,xgb"   # model families to train; finished models are skipped (svm: see the Extra section)
HISTORY_FAMILIES = "logreg,pcamon,rf,xgb"    # families also trained with 30-minute history inputs ("" = none)
HIST_WINDOWS = "2,5,20,40"                   # window sweep in 3-min samples (6 min, 15 min, 1 h, 2 h) for the best history models ("" = skip)
FORCE_RETRAIN = False                        # True = retrain the families in FAMILIES even if already finished

In [ ]:
# Train and validate every model. Finished models are skipped and older exports are refreshed from the saved
# models without retraining, so re-run this cell after a disconnect.
force = "--force-retrain" if FORCE_RETRAIN else ""
!python -u tep_pipeline.py --data-dir "{DATA_DIR}" --work-dir /content/dataset --runs-per-fault {RUNS_PER_FAULT} --families {FAMILIES} --history-families "{HISTORY_FAMILIES}" --hist-windows "{HIST_WINDOWS}" {force}

In [ ]:
# Results: the best model of each family and input set
import json
from pathlib import Path
import pandas as pd

out = Path(DATA_DIR) / ("outputs_full" if RUNS_PER_FAULT == 0 else f"outputs_{RUNS_PER_FAULT}runs") / "web_data"
manifest = json.loads((out / "manifest.json").read_text())

for task, key in [("detection", "recall"), ("diagnosis", "macro_F1")]:
    table = pd.DataFrame([{"model": m["label"], "family": m.get("family", "logreg"), "inputs": m.get("features", "inst"),
                           "used": m["coef_nonzero"], **{k: v for k, v in m["val"].items() if k not in ("TN", "FP", "FN", "TP")}}
                          for m in manifest[task]["models"]]).set_index("model")
    print(f"\n{task.upper()}  (best: {manifest[task]['best']}, rule: {manifest[task]['selection_rule']})")
    display(table.loc[table.groupby(["family", "inputs"])[key].idxmax()].sort_values(key, ascending=False).round(4))

In [ ]:
# Final test: run ONCE, after you are done choosing models
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    !python -u tep_pipeline.py --data-dir "{DATA_DIR}" --work-dir /content/dataset --runs-per-fault {RUNS_PER_FAULT} --families {FAMILIES} --history-families "{HISTORY_FAMILIES}" --hist-windows "{HIST_WINDOWS}" --final-test

In [ ]:
# Download the website files as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/web_data", "zip", out)
files.download("/content/web_data.zip")

## Extra: add the RBF SVM and the random-split demonstration

Trains **only** the RBF support vector machine (on small random subsets of the training rows) and the random-split
demonstration (logistic regression and a deep tree validated on random rows vs unseen runs), and adds them to the finished
full-data results; nothing else is retrained. Run **Setup** first (and keep `DATA_DIR` from **Settings**), then these cells.

- The finished results are taken from this repo's `docs/data` folder (the website's data), so nothing needs uploading.
- The first run converts the CSVs to a faster format (about 10 minutes). Then each SVM takes about 2–20 minutes, about 2 hours
  for all 16. Each finished SVM is saved to Drive, so after a disconnect just run the cells again.
- Learning curve, end-to-end and the final test are kept as they are. The SVM is never chosen as the best model.

In [ ]:
# Put the finished results in place (only if they aren't there yet, so a restart keeps finished SVMs)
import shutil
from pathlib import Path
full_web = Path(DATA_DIR) / "outputs_full" / "web_data"
if not (full_web / "manifest.json").exists():
    shutil.copytree("/content/tep-fault-detection/docs/data", full_web, dirs_exist_ok=True)
    print("Copied the finished results to", full_web)
else:
    print("Finished results already in place:", full_web)
print(len(list((full_web / "detection").glob("*.json"))), "detection files,",
      len(list((full_web / "diagnosis").glob("*.json"))), "diagnosis files")

In [ ]:
# Train the RBF SVMs (and the random-split demonstration, about 15 minutes). Everything else is reused: --history-families must list the same families as the full run, plus svm.
SVM_ROWS = "5000,20000"   # training-subset sizes; time grows with the square of these
SVM_C = "1,10"
!python -u tep_pipeline.py --data-dir "{DATA_DIR}" --work-dir /content/dataset --runs-per-fault 0 --families svm --history-families "logreg,pcamon,rf,xgb,svm" --hist-windows "2,5,20,40" --svm-rows "{SVM_ROWS}" --svm-c "{SVM_C}" --learning-curve 0 --end-to-end 0 --split-demo 1

In [ ]:
# Download the updated website files (put their contents in docs/data in the repo)
import shutil
from google.colab import files
shutil.make_archive("/content/web_data_svm", "zip", full_web)
files.download("/content/web_data_svm.zip")